<a href="https://colab.research.google.com/github/smathur2480/models_pub/blob/main/gen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install neurolib
!pip install matplotlib
!pip install scipy
!pip install vbi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.8/356.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.9/866.9 kB 14.7 MB/s eta 0:00:00


In [1]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import glob

from neurolib.models.wc import WCModel
import torch
import torch.nn as nn

import neurolib.utils.loadData as ld
import neurolib.utils.functions as func
from vbi.models.numba.bold import ParBold, do_bold_step

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")


outputs_exc  = []
outputs_bold = []
batch_size   = 5000



theta_lower_alpha = 0.24
theta_upper_alpha = 0.40
theta_lower_tau   = 1.6
theta_upper_tau   = 2.4
theta_lower_eo    = 0.2
theta_upper_eo    = 0.55



# sample each parameter independently from uniform distributions
alpha_inputs = np.random.uniform(theta_lower_alpha, theta_upper_alpha, batch_size)
tau_inputs   = np.random.uniform(theta_lower_tau,   theta_upper_tau,   batch_size)
eo_inputs    = np.random.uniform(theta_lower_eo,    theta_upper_eo,    batch_size)

print("batches were set up!")

# stack into (batch_size, 3) tensor for SBI — [alpha, tau, Eo]
raw_theta_tensor = torch.tensor(
    np.stack([alpha_inputs, tau_inputs, eo_inputs], axis=1),
    dtype=torch.float32
)

print("torch coversion done of theta inputs!")

from joblib import Parallel, delayed
import multiprocessing

def run_single_sim(i):
    model = WCModel()
    model.params['duration'] = 15 * 60000
    model.params['sigma_ou'] = 0.01
    model.run()
    exc = model.outputs['exc'][0]

    dtt          = model.params['dt'] / 1000.0
    steps_per_tr = max(1, int(round(2000.0 / model.params['dt'])))

    P = ParBold(alpha=alpha_inputs[i], tau=tau_inputs[i], Eo=eo_inputs[i])

    k1 = 4.3 * P.theta0 * P.Eo * P.TE
    k2 = P.epsilon * P.r0 * P.Eo * P.TE
    k3 = 1.0 - P.epsilon

    nn_val = 1
    s = np.ones((2, nn_val));  f = np.ones((2, nn_val))
    ftilde = np.zeros((2, nn_val)); vtilde = np.zeros((2, nn_val))
    qtilde = np.zeros((2, nn_val)); v = np.ones((2, nn_val))
    q = np.ones((2, nn_val))

    bold_out = []
    for j, x in enumerate(exc):
        r_in = np.array([x])
        do_bold_step(r_in, s, f, ftilde, vtilde, qtilde, v, q, dtt, P)
        if (j % steps_per_tr) == 0:
            bold_val = P.vo * (
                k1 * (1.0 - q[0, 0])
              + k2 * (1.0 - q[0, 0] / v[0, 0])
              + k3 * (1.0 - v[0, 0])
            )
            bold_out.append(bold_val)

    bold = np.array(bold_out)
    cutoff_exc  = len(exc)  // 3
    cutoff_bold = len(bold) // 3
    return exc[-cutoff_exc:], bold[-cutoff_bold:]

# Use all available cores (you have 8 with -n 8)
n_jobs = multiprocessing.cpu_count()
print(f"Running {batch_size} sims across {n_jobs} cores...")

results = Parallel(n_jobs=n_jobs, backend="loky", verbose=10)(
    delayed(run_single_sim)(i) for i in range(batch_size)
)

outputs_exc, outputs_bold = zip(*results)
outputs_exc  = list(outputs_exc)
outputs_bold = list(outputs_bold)

print('simulations are done')

from sbi.inference import NPE
from sbi.analysis import ActiveSubspace
from torch.distributions import Uniform

from torch.distributions import Independent, Uniform

prior = Independent(
    Uniform(
        low  = torch.tensor([theta_lower_alpha, theta_lower_tau, theta_lower_eo], device=device),
        high = torch.tensor([theta_upper_alpha, theta_upper_tau, theta_upper_eo], device=device)
    ),
    reinterpreted_batch_ndims=1
)

array_output_bold = np.array(outputs_bold)
x_tensor = torch.from_numpy(array_output_bold).float()   # shape (batch_size, n_bold_timepoints)

inference = NPE(prior, device=str(device))  # add device here
_ = inference.append_simulations(raw_theta_tensor, x_tensor).train()

observed_bold = x_tensor.mean(dim=0)
posterior = inference.build_posterior().set_default_x(observed_bold)

sensitivity = ActiveSubspace(posterior)
e_vals, e_vecs = sensitivity.find_directions(posterior_log_prob_as_property=True)

print("Eigenvalues: \n", e_vals, "\n")
print("Eigenvectors: \n", e_vecs)

from sbi.analysis import ActiveSubspace, pairplot
posterior_samples = posterior.sample((5000,)).cpu()
_ = pairplot(posterior_samples, limits=[[theta_lower_alpha, theta_upper_alpha], [theta_lower_tau, theta_upper_tau], [theta_lower_eo, theta_upper_eo]], figsize=(4, 4))
plt.savefig("/work/sm222/pairplot.png", dpi=150, bbox_inches="tight")

# e_vals shape: (n_params,)       = (3,)
# e_vecs shape: (n_params, n_params) = (3, 3)
# e_vecs columns are eigenvectors, so e_vecs[:, j] is the jth eigenvector

# activity score for parameter i:
# alpha_i = sum over j of (lambda_j * w_ij^2)
# where lambda_j is the jth eigenvalue and w_ij is the ith component of the jth eigenvector

# ── activity scores ───────────────────────────────────────────────────────────
# move to CPU first for consistency
e_vals = e_vals.cpu()
e_vecs = e_vecs.cpu()

activity_scores = torch.zeros(3)
for i in range(3):
    for j in range(3):
        activity_scores[i] += e_vals[j] * e_vecs[i, j] ** 2

param_names = ['alpha', 'tau', 'Eo']

print("Activity Scores:")
for name, score in zip(param_names, activity_scores):
    print(f"  {name}: {score.item():.6e}")

ranked = sorted(zip(param_names, activity_scores.tolist()), key=lambda x: x[1], reverse=True)
print("\nRanked by importance:")
for rank, (name, score) in enumerate(ranked, 1):
    print(f"  {rank}. {name}: {score:.6e}")


ModuleNotFoundError: No module named 'neurolib'